In [10]:
import arcpy
import os
arcpy.env.overwriteOutput        = True
arcpy.env.outputCoordinateSystem = None
arcpy.CheckOutExtension("Spatial")
project_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/Wetland Mapping"
wetland_gdb = os.path.normpath(os.path.join(project_folder, "Illinois_Wetland_Shapefiles.gdb"))
census_gdb = os.path.normpath(os.path.join(project_folder, "TIGER_Line_Census.gdb"))
target_sr = arcpy.SpatialReference(102003) # target coordinate system: NAD 1983 Contiguous USA Albers — WKID 102003

In [11]:
# output nondeepwater layer
step4 = os.path.normpath(os.path.join(wetland_gdb, "IL_WS_Step4_NonDeep"))

# Prepare NLCD rasters

In [3]:
years = [2023, 2024]
n_y = len(years)

In [4]:
rasters_wgs1984 = [f"F:/databases/NLCD/Land_Cover_Change.gdb/NLCD_{y}_IL_Box_Clip_WGS1984" for y in years]     

In [5]:
rasters_nad1983albers = [f"F:/databases/NLCD/Land_Cover_Change.gdb/NLCD_{y}_IL_Box_Clip_AlbersNAD1983" for y in years]     

In [6]:
raster_polygons_wgs1984 = [f"F:/databases/NLCD/Land_Cover_Change.gdb/NLCD_{y}_IL_Polygons_WGS1984" for y in years]

In [7]:
raster_polygons_nad1983_albers = [f"F:/databases/NLCD/Land_Cover_Change.gdb/NLCD_{y}_IL_Polygons_AlbersNAD1983" for y in years]

In [8]:
## loop through years and project polygons
#for i in range(n_y):
#    # projection with datum transform
#    arcpy.management.Project(
#        in_dataset=raster_polygons_wgs1984[i],
#        out_dataset=raster_polygons_nad1983_albers[i],
#        out_coor_system=target_sr,
        #transform_method= "WGS_1984_(ITRF00)_To_NAD_1983" #"NAD_1983_To_WGS_1984_1"
#    );
    
#    # CRS check
#    sr1 = arcpy.Describe(step4).spatialReference
#    sr2 = arcpy.Describe(raster_polygons_nad1983_albers[i]).spatialReference
#    print(f"Layer 1 CRS: {sr1.name}")
#    print(f"Layer 1 CRS: {sr1.factoryCode}")
#    print(f"Layer 2 CRS: {sr2.name}") 
#    print(f"Layer 2 CRS: {sr2.factoryCode}")

#    # polygon count check
#    int(arcpy.management.GetCount(raster_polygons_wgs1984[i])[0]) == int(arcpy.management.GetCount(raster_polygons_nad1983_albers[i])[0])

In [9]:
# project rasters
for i in range(n_y):
    arcpy.management.ProjectRaster(
        in_raster=rasters_wgs1984[i],
        out_raster=rasters_nad1983albers[i],
        out_coor_system=target_sr,
        resampling_type="NEAREST"       
    );

In [12]:
# loop through years and convert rasters to polygons
for i in range(n_y):
    arcpy.conversion.RasterToPolygon(
        in_raster=rasters_nad1983albers[i],
        out_polygon_features=raster_polygons_nad1983_albers[i], #raster_polygons_wgs1984[i],
        simplify="NO_SIMPLIFY",          
        raster_field="VALUE",         
        create_multipart_features="SINGLE_OUTER_PART",
        max_vertices_per_feature=None
    )
    count = int(arcpy.management.GetCount(raster_polygons_nad1983_albers[i])[0])
    print(f"Raster converted: {count} polygons")

Raster converted: 9803413 polygons
Raster converted: 9964202 polygons


# Intersect non-deepwater NWI with NLCD

In [13]:
# name of NWI x NLCD intersected layer (using step 6 because previously step 5 was joining with NWI difference product)
step6 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step6_NonDeep_{y}NLCD_Intersect")) for y in years]

In [14]:
# intersect non-deepwater NWI files with NLCD
for i in range(n_y):
    arcpy.analysis.PairwiseIntersect(
        in_features=[step4, raster_polygons_nad1983_albers[i]],
        out_feature_class=step6[i],
        join_attributes="ALL",
        output_type="INPUT"
    );

# Remove cropped and medium-to-high intensity polygon inclusions

In [15]:
# define new output layers
step7 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step7_{y}NLCD_LandCover")) for y in years]

In [16]:
# remove polygon inclusions
for i in range(n_y):
    # export all but row crops (82), medium intensity development (23), and high intensity development (24)
    arcpy.conversion.ExportFeatures(
        in_features = step6[i],
        out_features = step7[i],
        where_clause = "gridcode NOT IN (23, 24, 82)"
    )

    # check selection count
    count6 = int(arcpy.management.GetCount(step6[i])[0])
    count7 = int(arcpy.management.GetCount(step7[i])[0])
    print(f"Previous number of features: {count6}")
    print(f"Selected number of features: {count7}")

Previous number of features: 1038049
Selected number of features: 847429
Previous number of features: 1043352
Selected number of features: 852353


In [17]:
dissolve_fields = [
    "NWI_ID",
    "ATTRIBUTE",
    "WETLAND_TY",
    "ACRES",
    "SYSTEM_NAM",
    "SYSTEM_DEF",
    "SUBSYSTEM_",
    "SUBSYSTEM1",
    "CLASS_NAME",
    "CLASS_DEFI",
    "SUBCLASS_N",
    "SUBCLASS_D",
    "SPLIT_CLAS",
    "SPLIT_CL_1",
    "SPLIT_SUBC",
    "SPLIT_SU_1",
    "WATER_REGI",
    "WATER_RE_1",
    "WATER_RE_2",
    "FIRST_MODI",
    "FIRST_MO_1",
    "FIRST_MO_2",
    "FIRST_MO_3",
    "SECOND_MOD",
    "SECOND_M_1",
    "SECOND_M_2"
]

In [18]:
# dissolved layer names
step8 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step8_{y}NLCD_Dissolve_LandCover")) for y in years]

In [19]:
# dissolve polygons to original NWI IDs
for i in range(n_y):
    existing_fields = [f.name for f in arcpy.ListFields(step7[i])]

    # shorten wetland type field name
    if 'WETLAND_TY' not in existing_fields:
        arcpy.management.AlterField(
            in_table=step7[i],
            field='WETLAND_TYPE',
            new_field_name='WETLAND_TY'
        );

    # dissolve
    arcpy.analysis.PairwiseDissolve(
        in_features=step7[i],
        out_feature_class=step8[i],
        dissolve_field=dissolve_fields,
        multi_part="MULTI_PART"
    );

    # check selection count
    count4 = int(arcpy.management.GetCount(step4)[0])
    count8 = int(arcpy.management.GetCount(step8[i])[0])
    print(f"Previous number of features: {count4}")
    print(f"Selected number of features: {count8}")

Previous number of features: 312421
Selected number of features: 266666
Previous number of features: 312421
Selected number of features: 267155


# Filter wetland polygons based on Cowardin modifiers

## First modifier: remove farmed and excavated wetlands

In [20]:
# define new output layers
step9 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step9_{y}NLCD_FirstModifier")) for y in years]

In [21]:
for i in range(n_y):
    # export non-farmed and non-excavated
    arcpy.conversion.ExportFeatures(
        in_features = step8[i],
        out_features = step9[i],
        where_clause = "FIRST_MODI IS NULL OR (FIRST_MODI NOT IN ('Farmed','Excavated'))"
    )

    # check selection count
    count8 = int(arcpy.management.GetCount(step8[i])[0])
    count9 = int(arcpy.management.GetCount(step9[i])[0])
    print(f"Previous number of features: {count8}")
    print(f"Selected number of features: {count9}")

Previous number of features: 266666
Selected number of features: 222147
Previous number of features: 267155
Selected number of features: 222494


## Water regime: remove artificially flooded wetlands

In [22]:
# define new output layers
step10 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step10_{y}NLCD_WaterRegime")) for y in years]

In [23]:
for i in range(n_y):
    # export non-artificially flooed
    arcpy.conversion.ExportFeatures(
        in_features = step9[i],
        out_features = step10[i],
        where_clause="WATER_REGI IS NULL OR WATER_REGI NOT IN ('Artificially Flooded')"
    );
    
    # check selection count
    count9 = int(arcpy.management.GetCount(step9[i])[0])
    count10 = int(arcpy.management.GetCount(step10[i])[0])
    print(f"Previous number of features: {count9}")
    print(f"Selected number of features: {count10}")

Previous number of features: 222147
Selected number of features: 221658
Previous number of features: 222494
Selected number of features: 222005


## Area: remove wetlands with area < 0.10 ac

### Re-calculate areas

In [24]:
for i in range(n_y):
    # planar area
    arcpy.management.CalculateGeometryAttributes(
        in_features=step10[i],
        geometry_property=[["Polygon_Area_Ha_Planar", "AREA"]],
        area_unit="HECTARES",
        coordinate_system=target_sr,  # NAD 1983 Albers
        coordinate_format="SAME_AS_INPUT"                  
    );

    # geodesic area
    arcpy.management.CalculateGeometryAttributes(
        in_features=step10[i],
        geometry_property=[["Polygon_Area_Ha_Geodesic", "AREA_GEODESIC"]],
        area_unit="HECTARES",
        coordinate_system=target_sr,
        coordinate_format="SAME_AS_INPUT"
    );

### Convert areas to acres

In [26]:
for i in range(n_y):
    # add fields for both area types
    for field, alias in [("Polygon_Area_Ac_Planar","Polygon_Area_Ac_Planar"), ("Polygon_Area_Ac_Geodesic","Polygon_Area_Ac_Geodesic")]:
        if field not in [f.name for f in arcpy.ListFields(step10[i])]:
            arcpy.management.AddField(step10[i], field, "DOUBLE")

    # planar area
    arcpy.management.CalculateField(
        in_table=step10[i],
        field="Polygon_Area_Ac_Planar",
        expression="!Polygon_Area_Ha_Planar! * 2.47105",
        expression_type="PYTHON3"
    );

    # geodesic area
    arcpy.management.CalculateField(
        in_table=step10[i],
        field="Polygon_Area_Ac_Geodesic",
        expression="!Polygon_Area_Ha_Geodesic! * 2.47105", 
        expression_type="PYTHON3"
    );

### Remove categorically excluded small polygons

In [27]:
# define new output layers
step11 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step11_{y}NLCD_AreaThreshold")) for y in years]

In [28]:
for i in range(n_y):
    # export all wetlands with area >= 0.10 acre
    arcpy.conversion.ExportFeatures(
        in_features = step10[i],
        out_features = step11[i],
        where_clause="Polygon_Area_Ac_Geodesic >= 0.10"
    );

    # check selection count
    count10 = int(arcpy.management.GetCount(step10[i])[0])
    count11 = int(arcpy.management.GetCount(step11[i])[0])
    print(f"Previous number of features: {count10}")
    print(f"Selected number of features: {count11}")

Previous number of features: 221658
Selected number of features: 198509
Previous number of features: 222005
Selected number of features: 198859


In [29]:
# compare estimated wetland areas
for i in range(n_y):
    total = sum(
        row[0] for row in arcpy.da.SearchCursor(step11[i], ["Polygon_Area_Ha_Geodesic"])
        if row[0] is not None
    )
    
    print(f"Total {years[i]} wetland area: {total:,.2f}")

Total 2023 wetland area: 394,925.63
Total 2024 wetland area: 395,224.28
